In [1]:
import pandas as pd
import numpy as np
from lets_plot import *

# import data

In [2]:
%ls

close_price_panel_full_unbalanced.csv  sma_backtest.ipynb
regression_df.csv                      sma_backtest_fixed.ipynb


In [3]:
crypto = pd.read_csv(
    "close_price_panel_full_unbalanced.csv", 
    parse_dates=True, 
    index_col="date"
)

crypto

,ADA,BNB,BTC,DOGE,ETH,LINK,SOL,TRX,XRP,ZEC
date,,,,,,,,,,
2013-04-28,NaN,NaN,141.960000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-04-29,NaN,NaN,135.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-04-30,NaN,NaN,117.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-05-01,NaN,NaN,103.430000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-05-02,NaN,NaN,91.010000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2026-09-09,0.211898,722.659278,78283.060472,0.086159,2467.529168,11.804985,101.568283,0.338806,1.394741,1244.287571
2026-09-10,NaN,NaN,NaN,0.082878,NaN,11.466747,98.631258,0.339926,NaN,1079.814146
2026-09-11,NaN,NaN,NaN,0.084283,NaN,11.546361,102.394403,0.338532,NaN,1164.472000


Chosen based on longest available data, market cap

In [4]:
coins = (
    crypto
    [["BTC", "ETH", "XRP"]]
    .dropna()
)

coins

,BTC,ETH,XRP
date,,,
2015-08-07,259.800728,1.330750,0.008449
2015-08-08,264.338483,0.687586,0.008809
2015-08-10,269.866800,1.067379,0.008637
2015-08-11,267.712600,1.256613,0.008326
2015-08-12,263.655819,1.825395,0.008168
...,...,...,...
2026-09-05,79821.698732,2480.003169,1.412271
2026-09-06,80329.347846,2514.009269,1.422649
2026-09-07,79093.464190,2489.396754,1.396094


In [5]:
equity_coins =\
(
    coins
    .pct_change()
    .fillna(0)
    .add(1)
    .cumprod()
)

equity_coins

,BTC,ETH,XRP
date,,,
2015-08-07,1.000000,1.000000,1.000000
2015-08-08,1.017466,0.516690,1.042595
2015-08-10,1.038745,0.802088,1.022264
2015-08-11,1.030454,0.944289,0.985428
2015-08-12,1.014839,1.371704,0.966744
...,...,...,...
2026-09-05,307.242013,1863.613127,167.151624
2026-09-06,309.196007,1889.167213,168.379909
2026-09-07,304.438962,1870.671993,165.236936


In [6]:
equity_coins_LONG =\
(   
    equity_coins
    .reset_index()
    .melt(
        id_vars = "date",
        var_name = "coin",
        value_name = "equity"
    )
)

In [7]:
LetsPlot.setup_html()

In [8]:
plot_coins_log =\
(
    ggplot(
        equity_coins_LONG,
        aes(x="date", y="equity")
    )

    + geom_line(aes(color="coin"))
    + scale_y_log10()
)

plot_coins =\
(
    ggplot(
        equity_coins_LONG,
        aes(x="date", y="equity")
    )

    + geom_line(aes(color="coin"))
)

In [9]:
(
    gggrid([plot_coins_log, plot_coins], ncol=1)
    + ggsize(2000,1000) 
    + labs(title="BTC, ETH, and XRP growth—visualised on log and normal scale")
)

In [10]:
coins_returns =\
(
    coins
    .pct_change()
    .fillna(0.0)
)

coins_returns.corr()

,BTC,ETH,XRP
BTC,1.000000,0.594699,0.418612
ETH,0.594699,1.000000,0.395137
XRP,0.418612,0.395137,1.000000


In [11]:
coins_returns_log =\
(
    np.log(
        coins
        .div(coins.shift(1))
    )
)

coins_returns_log

,BTC,ETH,XRP
date,,,
2015-08-07,NaN,NaN,NaN
2015-08-08,0.017316,-0.660311,0.041713
2015-08-10,0.020698,0.439775,-0.019693
2015-08-11,-0.008014,0.163214,-0.036699
2015-08-12,-0.015269,0.373377,-0.019143
...,...,...,...
2026-09-05,0.001886,0.009691,0.009660
2026-09-06,0.006340,0.013619,0.007321
2026-09-07,-0.015505,-0.009838,-0.018842


In [12]:
coins_corr_long =\
(
    coins_returns
    .corr()
    .rename_axis(index = "coin_x")
    .reset_index()
    .melt(
        id_vars= "coin_x",
        var_name= "coin_y",
        value_name= "correlation"
    )
)

coins_corr_long

,coin_x,coin_y,correlation
0,BTC,BTC,1.000000
1,ETH,BTC,0.594699
2,XRP,BTC,0.418612
3,BTC,ETH,0.594699
4,ETH,ETH,1.000000
5,XRP,ETH,0.395137
6,BTC,XRP,0.418612
7,ETH,XRP,0.395137
8,XRP,XRP,1.000000


In [13]:
(
    ggplot(
        coins_corr_long.round(2),
        aes(x="coin_x", y="coin_y", fill="correlation")
    )

    + geom_tile()
    + labs(x="", y="")
    + geom_text(aes(label="correlation"), color = "white", size=10)
)

# Setting parameters

In [14]:
sma_parameters = [(3, 10), (5, 20), (10, 50), (20, 100), (50, 200)]

In [15]:
strategy_order = []

for f, s  in sma_parameters:
    strategy_order.append(f"SMA ({f}, {s})")



In [16]:
%whos list

Variable         Type    Data/Info
----------------------------------
sma_parameters   list    n=5
strategy_order   list    n=5


Strategy Vectors:
- [x] price ($P_t$)
- [x] noise-reduced price ($\text{sma}_t$)
- [x] event signal ($E_t$)
- [x] postion ($Q_t$)
- [x] pnl ($\Pi_t$)

In [17]:
coins

,BTC,ETH,XRP
date,,,
2015-08-07,259.800728,1.330750,0.008449
2015-08-08,264.338483,0.687586,0.008809
2015-08-10,269.866800,1.067379,0.008637
2015-08-11,267.712600,1.256613,0.008326
2015-08-12,263.655819,1.825395,0.008168
...,...,...,...
2026-09-05,79821.698732,2480.003169,1.412271
2026-09-06,80329.347846,2514.009269,1.422649
2026-09-07,79093.464190,2489.396754,1.396094


# Signals

1. We create skeleton df (just index)
2. Run for loop: each pair, calculate signal, put them in the skeleton dataframe 

In [18]:
TRADING_DAYS = 365


def sharpe(pnl):
    """Annualised reward per unit of variability, risk-free rate taken as zero."""

    pnl = pd.Series(pnl).dropna()

    return np.sqrt(TRADING_DAYS) * pnl.mean() / pnl.std()


def total_return(pnl):
    """Log returns add, so the total is a sum brought back to a simple return."""

    return np.exp(pd.Series(pnl).dropna().sum()) - 1.0


def max_drawdown(pnl):
    """The deepest peak-to-trough fall the equity curve ever suffered."""

    equity =\
    (
        np
        .exp(pd.Series(pnl).dropna().cumsum())
    )

    return (equity / equity.cummax() - 1.0).min()


def cagr(pnl):
    """The smooth annual rate that produces the same end-to-end growth."""

    pnl = pd.Series(pnl).dropna()
    years = len(pnl) / TRADING_DAYS

    return np.exp(pnl.sum()) ** (1.0 / years) - 1.0


def trades(position):
    """How many times the held position actually changed."""

    return int(pd.Series(position).dropna().diff().abs().gt(0).sum())

def volatility(pnl):
    """Annualised standard deviation of the log return stream."""
    
    return np.sqrt(TRADING_DAYS) * pd.Series(pnl).dropna().std()


def scorecard(pnl, position = None):

    card = {
        "total_return": total_return(pnl),
        "annualised_volatility": volatility(pnl),
        "annualised_sharpe": sharpe(pnl),
        "max_drawdown": max_drawdown(pnl),
        "cagr": cagr(pnl),
        "days": len(pd.Series(pnl).dropna()),
    }

    if position is not None:
        card["trades"] = trades(position)

    return pd.Series(card)

In [19]:
COST_BPS = 10.0 # one-way cost in basis points; 10 bps = 0.10% per side

def momentum_book_qf627(price, fast, slow, cost_bps = COST_BPS):
    """Long-flat trend following. Hold the instrument while the fast average
    sits above the slow one, hold cash otherwise. Charges cost_bps per 
    side on every change in the held position."""

    book =\
    (
        pd
        .DataFrame({"price": price})
        .dropna()
    )

    book["ma_fast"] = book["price"].rolling(window = fast).mean() 
    book["ma_slow"] = book["price"].rolling(window = slow).mean()
    book["signal"] = np.where(book["ma_fast"] > book["ma_slow"], 1, 0)
    
    book["detected_position"] = book["signal"] 
    book["executed_position"] = book["detected_position"].shift(1)
    
    book["log_return"] = np.log(book["price"] / book["price"].shift(1))
    
    book["turnover"] = book["executed_position"].diff().abs() 
    book["cost"] = book["turnover"] * (cost_bps / 10_000.0)
    
    book["pnl_gross"] = book["executed_position"] * book["log_return"] 
    book["pnl"] = book["pnl_gross"] - book["cost"]

    return book.dropna()

In [20]:
pnl_strat =\
(
    pd.DataFrame(
        index = coins.index
    )
)

for coin in coins.columns:
    pnl_strat[f"{coin}_hold"] = coins_returns_log[coin]

    for f, s in sma_parameters:
        pnl_strat[f"{coin}_sma_{f}_{s}"] = momentum_book_qf627(coins[coin], fast=f, slow=s, cost_bps=COST_BPS).pnl

pnl_strat = pnl_strat.dropna()

In [21]:
pnl_strat.BTC_sma_3_10

date
2016-02-23    0.006841
2016-02-24   -0.000106
2016-02-25   -0.001000
2016-02-26    0.000000
2016-02-27    0.000000
                ...   
2026-09-05    0.001886
2026-09-06    0.006340
2026-09-07   -0.015505
2026-09-08   -0.008159
2026-09-09   -0.002140
Name: BTC_sma_3_10, Length: 3852, dtype: float64

In [22]:
pnl_strat

,BTC_hold,BTC_sma_3_10,BTC_sma_5_20,BTC_sma_10_50,BTC_sma_20_100,BTC_sma_50_200,ETH_hold,ETH_sma_3_10,ETH_sma_5_20,ETH_sma_10_50,ETH_sma_20_100,ETH_sma_50_200,XRP_hold,XRP_sma_3_10,XRP_sma_5_20,XRP_sma_10_50,XRP_sma_20_100,XRP_sma_50_200
date,,,,,,,,,,,,,,,,,,
2016-02-23,0.006841,0.006841,0.006841,0.006841,0.000000,0.000000,0.088986,0.088986,0.088986,0.088986,0.088986,0.000000,0.012345,0.000000,0.012345,0.012345,0.012345,0.000000
2016-02-24,-0.000106,-0.000106,-0.000106,-0.000106,-0.001106,-0.001106,-0.032830,-0.032830,-0.032830,-0.032830,-0.032830,-0.033830,-0.005219,-0.000000,-0.005219,-0.005219,-0.005219,-0.006219
2016-02-25,0.009144,-0.001000,0.009144,0.009144,0.009144,0.009144,-0.006575,-0.006575,-0.006575,-0.006575,-0.006575,-0.006575,0.002737,0.000000,-0.001000,0.002737,0.002737,0.002737
2016-02-26,0.011895,0.000000,0.011895,0.011895,0.011895,0.011895,0.075068,0.075068,0.075068,0.075068,0.075068,0.075068,-0.009362,-0.000000,-0.000000,-0.009362,-0.009362,-0.009362
2016-02-27,0.001522,0.000000,0.001522,0.001522,0.001522,0.001522,0.013059,0.013059,0.013059,0.013059,0.013059,0.013059,-0.010970,-0.000000,-0.000000,-0.010970,-0.010970,-0.010970
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-05,0.001886,0.001886,0.001886,0.001886,0.001886,0.000000,0.009691,0.000000,0.009691,0.009691,0.009691,0.009691,0.009660,0.008660,0.009660,0.009660,0.009660,0.000000
2026-09-06,0.006340,0.006340,0.006340,0.006340,0.006340,0.000000,0.013619,0.012619,0.013619,0.013619,0.013619,0.013619,0.007321,0.007321,0.007321,0.007321,0.007321,0.000000
2026-09-07,-0.015505,-0.015505,-0.015505,-0.015505,-0.015505,-0.000000,-0.009838,-0.009838,-0.009838,-0.009838,-0.009838,-0.009838,-0.018842,-0.018842,-0.018842,-0.018842,-0.018842,-0.000000


# Step 5: Backtest

In [23]:
coin_position =\
(
    pd.DataFrame(
        index = coins.index
    )
)

for coin in coins.columns:
    coin_position[f"{coin}_hold"] = momentum_book_qf627(coins[coin], fast=f, slow=s)["executed_position"]
    for f, s in sma_parameters:
        coin_position[f"{coin}_sma_{f}_{s}"] = momentum_book_qf627(coins[coin], fast=f, slow=s)["executed_position"]

coin_position = coin_position.dropna()

In [24]:
scorecard(pnl=pnl_strat["BTC_hold"], position=pnl_strat["BTC_hold"])

total_return              185.254313
annualised_volatility       0.663769
annualised_sharpe           0.746194
max_drawdown               -0.836457
cagr                        0.640991
days                     3852.000000
trades                   3851.000000
dtype: float64

# Run Fit-Test windows

In [25]:
import statsmodels.api as sm

def alpha_beta(strategy_log, market_log, rf_annual = 2.54/100):
    """Rp - Rf = alpha + beta (Rm - Rf) + eps, on log returns, with HAC errors."""
    
    rf = rf_annual / TRADING_DAYS
    
    data =\
    (
        pd .DataFrame({
            "excess_portfolio" : strategy_log - rf,
            "excess_market": market_log - rf,
        })
        .dropna() 
    )
    
    fit =\
    (
        sm
        .OLS(data["excess_portfolio"], sm.add_constant(data["excess_market"]))
        .fit(cov_type = "HAC", cov_kwds = {"maxlags": 5})
    )
    
    return pd.Series({ 
        "alpha_annual": fit.params["const"] * TRADING_DAYS, 
        "alpha_t"     : fit.tvalues["const"],
        "beta"        : fit.params["excess_market"],
        "r_squared"   : fit.rsquared,
        "n_da"        : int(fit.nobs),
    })

In [26]:
SPLIT_DATE = "2022-01-01"
METRIC_WEIGHTS = {"total_return": 0.3, "annualised_sharpe": 0.5, "max_drawdown": 0.2}
in_sample = pnl_strat.loc[pnl_strat.index < SPLIT_DATE] 
out_sample = pnl_strat.loc[pnl_strat.index >= SPLIT_DATE]

def select_pair(pnl_frame, coin):
    """Rank every SMA book for one coin over the given window, return the winner."""
    books = [c for c in pnl_frame.columns if c.startswith(f"{coin}_sma_")]
    board =\
    (
        pd
        .DataFrame({b: scorecard(pnl_frame[b]) for b in books}) 
        .T
        [list(METRIC_WEIGHTS.keys())]
        .rank(ascending = True)
    )
    winner = (board * pd.Series(METRIC_WEIGHTS)).sum(axis = 1).idxmax() 
    fast, slow = winner.replace(f"{coin}_sma_", "").split("_")
    
    return int(fast), int(slow)

best_pairs = {coin: select_pair(in_sample, coin) for coin in coins.columns} 
best_pairs

{'BTC': (5, 20), 'ETH': (5, 20), 'XRP': (3, 10)}

# Portfolio Construction

In [27]:
final_strat=\
(
    pnl_strat[[f"{coin}_sma_{best_pairs[coin][0]}_{best_pairs[coin][1]}" for coin in coins.columns] + ["BTC_hold", "ETH_hold", "XRP_hold"]].loc[SPLIT_DATE:]
    .assign(ew_portolio_log = np.log(np.exp(pnl_strat[[f"{coin}_sma_{best_pairs[coin][0]}_{best_pairs[coin][1]}" for coin in coins.columns]]).mean(axis=1)))
    .assign(crypto_market = np.log(np.exp(pnl_strat[["BTC_hold", "ETH_hold", "XRP_hold"]]).mean(axis=1)))
)

# Alpha Regression

In [28]:
regression_df =\
(
    final_strat[["ew_portolio_log", "crypto_market"]]
)

regression_df

,ew_portolio_log,crypto_market
date,,
2022-01-01,-0.000333,0.027191
2022-01-02,0.000000,0.004847
2022-01-03,0.000000,-0.021136
2022-01-04,0.000000,-0.006882
2022-01-05,0.000000,-0.060301
...,...,...
2026-09-05,0.006752,0.007086
2026-09-06,0.009099,0.009099
2026-09-07,-0.014722,-0.014722


In [29]:
regression_df_equity =\
(
    regression_df
    .cumsum()
    .pipe(np.exp)
    .reset_index()
    .melt(
        id_vars = "date",
        var_name = "book",
        value_name = "equity"
    )
)

regression_df_equity

,date,book,equity
0,2022-01-01,ew_portolio_log,0.999667
1,2022-01-02,ew_portolio_log,0.999667
2,2022-01-03,ew_portolio_log,0.999667
3,2022-01-04,ew_portolio_log,0.999667
4,2022-01-05,ew_portolio_log,0.999667
...,...,...,...
3421,2026-09-05,crypto_market,1.593953
3422,2026-09-06,crypto_market,1.608521
3423,2026-09-07,crypto_market,1.585015
3424,2026-09-08,crypto_market,1.587823


In [30]:
(
    ggplot(
        regression_df_equity,
        aes(x="date", y="equity")
    )

    + geom_line(aes(color="book"))
    + ggsize(2000, 1000)
)

In [31]:
regression_df.to_csv("regression_df.csv")

In [32]:
regression_df

,ew_portolio_log,crypto_market
date,,
2022-01-01,-0.000333,0.027191
2022-01-02,0.000000,0.004847
2022-01-03,0.000000,-0.021136
2022-01-04,0.000000,-0.006882
2022-01-05,0.000000,-0.060301
...,...,...
2026-09-05,0.006752,0.007086
2026-09-06,0.009099,0.009099
2026-09-07,-0.014722,-0.014722


In [33]:
COST_GRID = [0.0, 5.0, 10.0, 25.0, 50.0]
BEST_PAIRS = best_pairs
market_log = regression_df.crypto_market
cost_sensitivity = {}

for c in COST_GRID:
    pnl_c =\
    (
        pd .DataFrame({
            coin: momentum_book_qf627(coins[coin], * BEST_PAIRS[coin], cost_bps = c)["pnl"]
            for coin in coins.columns 
        })
        .dropna() 
    )
    
    ew_c = pnl_c.pipe(np.exp).mean(axis = 1).pipe(np.log).loc[SPLIT_DATE:]
    cost_sensitivity[f"{c:.0f} bps"] =\
    (
        pd .concat([
            scorecard(ew_c),
            alpha_beta(ew_c, market_log.reindex(ew_c.index)), 
        ])
    )

cost_sensitivity = pd.DataFrame(cost_sensitivity) 

cost_sensitivity

,0 bps,5 bps,10 bps,25 bps,50 bps
total_return,0.617151,0.510255,0.410452,0.149038,-0.183167
annualised_volatility,0.349855,0.349859,0.349870,0.349945,0.350211
annualised_sharpe,0.292746,0.251092,0.209447,0.084589,-0.123097
max_drawdown,-0.484019,-0.490629,-0.497153,-0.516217,-0.550680
cagr,0.107847,0.091821,0.076031,0.030044,-0.042194
days,1713.000000,1713.000000,1713.000000,1713.000000,1713.000000
alpha_annual,0.047424,0.032854,0.018289,-0.025381,-0.098081
alpha_t,0.386194,0.266958,0.148277,-0.204371,-0.780450
beta,0.414908,0.414875,0.414841,0.414740,0.414573
r_squared,0.502647,0.502555,0.502443,0.501983,0.500816


# Re-regression

In [34]:
alpha_beta(regression_df.ew_portolio_log, regression_df.crypto_market)

alpha_annual       0.018289
alpha_t            0.148277
beta               0.414841
r_squared          0.502443
n_da            1713.000000
dtype: float64

In [35]:
oos_pnl =\
(
    pd .DataFrame({
        coin: momentum_book_qf627(coins[coin], * best_pairs[coin])["pnl"]
        for coin in coins.columns 
    })
    .dropna() 
)

windows = {
    "in_sample" : in_sample.index, 
    "out_sample" : out_sample.index, 
    "full_sample" : pnl_strat.index,
}

alpha_table = {}

In [36]:
alpha_table

{}

In [37]:
?? alpha_beta

Signature:  alpha_beta(strategy_log, market_log, rf_annual=0.0254)
Source:   
def alpha_beta(strategy_log, market_log, rf_annual = 2.54/100):
    """Rp - Rf = alpha + beta (Rm - Rf) + eps, on log returns, with HAC errors."""
    
    rf = rf_annual / TRADING_DAYS
    
    data =\
    (
        pd .DataFrame({
            "excess_portfolio" : strategy_log - rf,
            "excess_market": market_log - rf,
        })
        .dropna() 
    )
    
    fit =\
    (
        sm
        .OLS(data["excess_portfolio"], sm.add_constant(data["excess_market"]))
        .fit(cov_type = "HAC", cov_kwds = {"maxlags": 5})
    )
    
    return pd.Series({ 
        "alpha_annual": fit.params["const"] * TRADING_DAYS, 
        "alpha_t"     : fit.tvalues["const"],
        "beta"        : fit.params["excess_market"],
        "r_squared"   : fit.rsquared,
        "n_da"        : int(fit.nobs),
    })
File:      /var/folders/sm/3jx63l6161xckxcwmjzwpr0r0000gn/T/ipykernel_89949/1200754878.py
Type:      funct

In [38]:
market_log

date
2022-01-01    0.027191
2022-01-02    0.004847
2022-01-03   -0.021136
2022-01-04   -0.006882
2022-01-05   -0.060301
                ...   
2026-09-05    0.007086
2026-09-06    0.009099
2026-09-07   -0.014722
2026-09-08    0.001770
2026-09-09   -0.008403
Name: crypto_market, Length: 1713, dtype: float64

In [39]:
?? pd.Series.reindex

Signature:
 pd.Series.reindex(
    self,
    index=None,
    *,
    axis: 'Axis | None' = None,
    method: 'str | None' = None,
    copy: 'bool | None' = None,
    level: 'Level | None' = None,
    fill_value: 'Scalar | None' = None,
    limit: 'int | None' = None,
    tolerance=None,
) -> 'Series'
Docstring:
Conform Series to new index with optional filling logic.

Places NA/NaN in locations having no value in the previous index. A new object
is produced unless the new index is equivalent to the current one and
``copy=False``.

Parameters
----------

index : array-like, optional
    New labels for the index. Preferably an Index object to avoid
    duplicating data.
axis : int or str, optional
    Unused.
method : {None, 'backfill'/'bfill', 'pad'/'ffill', 'nearest'}
    Method to use for filling holes in reindexed DataFrame.
    Please note: this is only applicable to DataFrames/Series with a
    monotonically increasing/decreasing index.

    * None (default): don't fill gaps
    * p

In [40]:
market_3 = np.log(np.exp(pnl_strat[["BTC_hold", "ETH_hold", "XRP_hold"]]).mean(axis=1))

In [41]:
for name, idx in windows.items():
    ew =\
    (
        oos_pnl
        .reindex(idx)
        .dropna()
        .pipe(np.exp)
        .mean(axis = 1)
        .pipe(np.log)
    )

    print(ew)
    
    alpha_table[name] =\
    (
        pd 
        .concat([
            scorecard(ew),
            alpha_beta(ew, market_3.reindex(ew.index)),
        ])
    )

alpha_table = pd.DataFrame(alpha_table) 

alpha_table

date
2016-02-23    0.032767
2016-02-24   -0.010860
2016-02-25    0.000877
2016-02-26    0.029534
2016-02-27    0.004877
                ...   
2021-12-27   -0.001041
2021-12-28   -0.067262
2021-12-29   -0.021780
2021-12-30    0.004168
2021-12-31   -0.005837
Length: 2139, dtype: float64
date
2022-01-01   -0.000333
2022-01-02    0.000000
2022-01-03    0.000000
2022-01-04    0.000000
2022-01-05    0.000000
                ...   
2026-09-05    0.006752
2026-09-06    0.009099
2026-09-07   -0.014722
2026-09-08    0.001770
2026-09-09   -0.008403
Length: 1713, dtype: float64
date
2016-02-23    0.032767
2016-02-24   -0.010860
2016-02-25    0.000877
2016-02-26    0.029534
2016-02-27    0.004877
                ...   
2026-09-05    0.006752
2026-09-06    0.009099
2026-09-07   -0.014722
2026-09-08    0.001770
2026-09-09   -0.008403
Length: 3852, dtype: float64


,in_sample,out_sample,full_sample
total_return,2465.215976,0.410452,3477.478769
annualised_volatility,0.602704,0.349870,0.507111
annualised_sharpe,2.211330,0.209447,1.523676
max_drawdown,-0.438513,-0.497153,-0.514416
cagr,2.791559,0.076031,1.165548
days,2139.000000,1713.000000,3852.000000
alpha_annual,0.739158,0.018289,0.432744
alpha_t,4.554622,0.148277,4.113276
beta,0.520186,0.414841,0.492767
r_squared,0.580297,0.502443,0.559053


In [42]:
metrics_fit=\
(
    scorecard(
        pnl_strat
        .loc[:SPLIT_DATE, pnl_strat.columns[0]]
    )
    .rename(pnl_strat.columns[0])
    .to_frame()
)

for col in pnl_strat.columns:
    metrics_fit[col] =\
    (
        scorecard(
            pnl_strat
            .loc[:SPLIT_DATE, col]
        )
    )

metrics_fit_LONG =\
(
    metrics_fit
    .rename_axis(index="metrics")
    .reset_index()
    .melt(
        id_vars = "metrics",
        var_name = "book"
    )
    .query(
        "['total_return', 'annualised_sharpe', 'max_drawdown'] in metrics"
    )
)

metrics_fit_LONG

,metrics,book,value
0,total_return,BTC_hold,112.743863
2,annualised_sharpe,BTC_hold,1.052639
3,max_drawdown,BTC_hold,-0.836457
6,total_return,BTC_sma_3_10,52.260176
8,annualised_sharpe,BTC_sma_3_10,1.255921
9,max_drawdown,BTC_sma_3_10,-0.510195
12,total_return,BTC_sma_5_20,104.314395
14,annualised_sharpe,BTC_sma_5_20,1.455036
15,max_drawdown,BTC_sma_5_20,-0.638598
18,total_return,BTC_sma_10_50,101.215118


In [43]:
BTC_metrics =\
(
    metrics_fit_LONG.loc[
        metrics_fit_LONG
        .book
        .str
        .startswith("BTC")
    ]
)

In [44]:
(
    ggplot(
        BTC_metrics, 
        aes(x="book", y="value", fill="book")
    )
    
    + geom_bar(
        stat="identity", 
        position=position_dodge(width=0.9), 
        width=0.8, 
        color="white"
    )

    + facet_wrap("metrics", scales="free_y")
    
    + scale_fill_manual(
        values=[
            "#4C72B0",  # steel blue
            "#DD8452",  # muted orange
            "#55A868",  # muted green
            "#8172B2",  # muted purple
            "#937860",  # muted brown
            "#64B5CD",  # dusty teal
        ]
    )
    
    + labs(
        title="Backtested Strategy Metrics by Parameter — BTC",
        x="Metrics",
        y="Value",
        fill="Strategy Version",
    )
    
    + theme_minimal()
    + theme(axis_text_x=element_text(angle=10, hjust=1))
    + ggsize(800, 1000)
)

In [45]:
ETH_metrics =\
(
    metrics_fit_LONG.loc[
        metrics_fit_LONG
        .book
        .str
        .startswith("ETH")
    ]
)

In [46]:
(
    ggplot(
        ETH_metrics, 
        aes(x="book", y="value", fill="book")
    )
    
    + geom_bar(
        stat="identity", 
        position=position_dodge(width=0.9), 
        width=0.8, 
        color="white"
    )

    + facet_wrap("metrics", scales="free_y")
    
    + scale_fill_manual(
        values=[
            "#4C72B0",  # steel blue
            "#DD8452",  # muted orange
            "#55A868",  # muted green
            "#8172B2",  # muted purple
            "#937860",  # muted brown
            "#64B5CD",  # dusty teal
        ]
    )
    
    + labs(
        title="Backtested Strategy Metrics by Parameter — ETH",
        x="Metrics",
        y="Value",
        fill="Strategy Version",
    )
    
    + theme_minimal()
    + theme(axis_text_x=element_text(angle=10, hjust=1))
    + ggsize(800, 1000)  # wider than your current 800
)

In [47]:
XRP_metrics =\
(
    metrics_fit_LONG.loc[
        metrics_fit_LONG
        .book
        .str
        .startswith("XRP")
    ]
)

In [48]:
(
    ggplot(
        XRP_metrics, 
        aes(x="book", y="value", fill="book")
    )
    
    + geom_bar(
        stat="identity", 
        position=position_dodge(width=0.9), 
        width=0.8, 
        color="white"
    )

    + facet_wrap("metrics", scales="free_y")
    
    + scale_fill_manual(
        values=[
            "#4C72B0",  # steel blue
            "#DD8452",  # muted orange
            "#55A868",  # muted green
            "#8172B2",  # muted purple
            "#937860",  # muted brown
            "#64B5CD",  # dusty teal
        ]
    )
    
    + labs(
        title="Backtested Strategy Metrics by Parameter — XRP",
        x="Metrics",
        y="Value",
        fill="Strategy Version",
    )
    
    + theme_minimal()
    + theme(axis_text_x=element_text(angle=10, hjust=1))
    + ggsize(800, 1000)
)

## Final Scores

In [49]:
scoreboard_strat =\
(
    pd.DataFrame(
        index = scorecard(pnl=pnl_strat["BTC_hold"], position=pnl_strat["BTC_hold"]).index
    )
)

for book in pnl_strat.columns:
    scoreboard_strat[book] = scorecard(pnl=pnl_strat[book], position=pnl_strat[book])

In [50]:
scoreboard_rankings =\
(
    scoreboard_strat
    .T
    .sort_values(by="annualised_sharpe", ascending=False)
)

In [51]:
scoreboard_rankings

,total_return,annualised_volatility,annualised_sharpe,max_drawdown,cagr,days,trades
ETH_sma_5_20,2234.049444,0.693531,1.053681,-0.664778,1.076658,3852.0,2191.0
BTC_sma_10_50,187.271623,0.493765,1.005176,-0.750737,0.642667,3852.0,2266.0
BTC_sma_5_20,126.416503,0.464997,0.987803,-0.638598,0.583007,3852.0,2378.0
ETH_sma_10_50,1555.771772,0.708843,0.982575,-0.721648,1.006700,3852.0,2122.0
BTC_sma_20_100,192.640822,0.526305,0.948091,-0.635725,0.647049,3852.0,2285.0
ETH_sma_3_10,680.208361,0.683522,0.904397,-0.680347,0.855539,3852.0,2463.0
XRP_sma_3_10,3517.316831,0.857909,0.901905,-0.643039,1.167886,3852.0,2116.0
BTC_sma_3_10,69.315392,0.459051,0.877890,-0.595881,0.496301,3852.0,2520.0
BTC_sma_50_200,149.204086,0.556128,0.853970,-0.648646,0.607880,3852.0,2355.0
ETH_sma_20_100,826.969009,0.761867,0.835662,-0.721604,0.890163,3852.0,2223.0


# Archive

### BTC

In [52]:
scoreboard_BTC =\
(
    scoreboard_rankings
    .loc[
        scoreboard_rankings
        .index
        .str
        .startswith("BTC")
    ]
)
scoreboard_BTC

,total_return,annualised_volatility,annualised_sharpe,max_drawdown,cagr,days,trades
BTC_sma_10_50,187.271623,0.493765,1.005176,-0.750737,0.642667,3852.0,2266.0
BTC_sma_5_20,126.416503,0.464997,0.987803,-0.638598,0.583007,3852.0,2378.0
BTC_sma_20_100,192.640822,0.526305,0.948091,-0.635725,0.647049,3852.0,2285.0
BTC_sma_3_10,69.315392,0.459051,0.877890,-0.595881,0.496301,3852.0,2520.0
BTC_sma_50_200,149.204086,0.556128,0.853970,-0.648646,0.607880,3852.0,2355.0
BTC_hold,185.254313,0.663769,0.746194,-0.836457,0.640991,3852.0,3851.0


In [53]:
scoreboard_BTC_ranks = scoreboard_BTC.rank(ascending=True).astype(int)
scoreboard_BTC_ranks

,total_return,annualised_volatility,annualised_sharpe,max_drawdown,cagr,days,trades
BTC_sma_10_50,5,3,6,2,5,3,1
BTC_sma_5_20,2,2,5,4,2,3,4
BTC_sma_20_100,6,4,4,5,6,3,2
BTC_sma_3_10,1,1,3,6,1,3,5
BTC_sma_50_200,3,5,2,3,3,3,3
BTC_hold,4,6,1,1,4,3,6


In [54]:
w1,  w2, w3, w4 = 0.5, 0.3, 0.2, 0.0

BTC_weights =\
(
    pd
    .DataFrame(
        index = scoreboard_BTC.index,
        data = {
                "total_return"        : w1,
                "annualised_sharpe"   : w2,
                "max_drawdown"        : w3,
                "cagr"                : w4,
                "days"                : w4,
                "trades"              : w4,
             }
    )
)

BTC_weights

,total_return,annualised_sharpe,max_drawdown,cagr,days,trades
BTC_sma_10_50,0.5,0.3,0.2,0.0,0.0,0.0
BTC_sma_5_20,0.5,0.3,0.2,0.0,0.0,0.0
BTC_sma_20_100,0.5,0.3,0.2,0.0,0.0,0.0
BTC_sma_3_10,0.5,0.3,0.2,0.0,0.0,0.0
BTC_sma_50_200,0.5,0.3,0.2,0.0,0.0,0.0
BTC_hold,0.5,0.3,0.2,0.0,0.0,0.0


In [55]:
(BTC_weights * scoreboard_BTC_ranks).sum(axis=1)

BTC_sma_10_50     4.7
BTC_sma_5_20      3.3
BTC_sma_20_100    5.2
BTC_sma_3_10      2.6
BTC_sma_50_200    2.7
BTC_hold          2.5
dtype: float64

### ETH

In [56]:
scoreboard_ETH =\
(
    scoreboard_rankings
    .loc[
        scoreboard_rankings
        .index
        .str
        .startswith("ETH")
    ]
)
scoreboard_ETH

,total_return,annualised_volatility,annualised_sharpe,max_drawdown,cagr,days,trades
ETH_sma_5_20,2234.049444,0.693531,1.053681,-0.664778,1.076658,3852.0,2191.0
ETH_sma_10_50,1555.771772,0.708843,0.982575,-0.721648,1.006700,3852.0,2122.0
ETH_sma_3_10,680.208361,0.683522,0.904397,-0.680347,0.855539,3852.0,2463.0
ETH_sma_20_100,826.969009,0.761867,0.835662,-0.721604,0.890163,3852.0,2223.0
ETH_sma_50_200,319.932824,0.774737,0.705863,-0.879649,0.727817,3852.0,2217.0
ETH_hold,438.680856,0.937184,0.615343,-0.942160,0.780136,3852.0,3851.0


In [57]:
scoreboard_ETH_ranks = scoreboard_ETH.rank(ascending=True).astype(int)
scoreboard_ETH_ranks

,total_return,annualised_volatility,annualised_sharpe,max_drawdown,cagr,days,trades
ETH_sma_5_20,6,2,6,6,6,3,2
ETH_sma_10_50,5,3,5,3,5,3,1
ETH_sma_3_10,3,1,4,5,3,3,5
ETH_sma_20_100,4,4,3,4,4,3,4
ETH_sma_50_200,1,5,2,2,1,3,3
ETH_hold,2,6,1,1,2,3,6


In [58]:
ETH_weights =\
(
    pd
    .DataFrame(
        index = scoreboard_ETH.index,
        data = {
                "total_return"        : w1,
                "annualised_sharpe"   : w2,
                "max_drawdown"        : w3,
                "cagr"                : w4,
                "days"                : w4,
                "trades"              : w4,
             }
    )
)

ETH_weights

,total_return,annualised_sharpe,max_drawdown,cagr,days,trades
ETH_sma_5_20,0.5,0.3,0.2,0.0,0.0,0.0
ETH_sma_10_50,0.5,0.3,0.2,0.0,0.0,0.0
ETH_sma_3_10,0.5,0.3,0.2,0.0,0.0,0.0
ETH_sma_20_100,0.5,0.3,0.2,0.0,0.0,0.0
ETH_sma_50_200,0.5,0.3,0.2,0.0,0.0,0.0
ETH_hold,0.5,0.3,0.2,0.0,0.0,0.0


### XRP

In [59]:
scoreboard_XRP =\
(
    scoreboard_rankings
    .loc[
        scoreboard_rankings
        .index
        .str
        .startswith("XRP")
    ]
)
scoreboard_XRP

,total_return,annualised_volatility,annualised_sharpe,max_drawdown,cagr,days,trades
XRP_sma_3_10,3517.316831,0.857909,0.901905,-0.643039,1.167886,3852.0,2116.0
XRP_sma_5_20,469.168496,0.833576,0.699447,-0.674000,0.791480,3852.0,1860.0
XRP_sma_10_50,159.373644,0.886211,0.542900,-0.909578,0.617892,3852.0,1831.0
XRP_hold,173.998828,1.123622,0.435550,-0.959186,0.631327,3852.0,3849.0
XRP_sma_20_100,4.868002,0.921122,0.182030,-0.990330,0.182549,3852.0,1811.0
XRP_sma_50_200,2.684658,0.949164,0.130197,-0.982241,0.131539,3852.0,1862.0


In [60]:
scoreboard_XRP_ranks = scoreboard_XRP.rank(ascending=True).astype(int)
scoreboard_XRP_ranks

,total_return,annualised_volatility,annualised_sharpe,max_drawdown,cagr,days,trades
XRP_sma_3_10,6,2,6,6,6,3,5
XRP_sma_5_20,5,1,5,5,5,3,3
XRP_sma_10_50,3,3,4,4,3,3,2
XRP_hold,4,6,3,3,4,3,6
XRP_sma_20_100,2,4,2,1,2,3,1
XRP_sma_50_200,1,5,1,2,1,3,4


In [61]:
XRP_weights =\
(
    pd
    .DataFrame(
        index = scoreboard_XRP.index,
        data = {
                "total_return"        : w1,
                "annualised_sharpe"   : w2,
                "max_drawdown"        : w3,
                "cagr"                : w4,
                "days"                : w4,
                "trades"              : w4,
             }
    )
)

XRP_weights

,total_return,annualised_sharpe,max_drawdown,cagr,days,trades
XRP_sma_3_10,0.5,0.3,0.2,0.0,0.0,0.0
XRP_sma_5_20,0.5,0.3,0.2,0.0,0.0,0.0
XRP_sma_10_50,0.5,0.3,0.2,0.0,0.0,0.0
XRP_hold,0.5,0.3,0.2,0.0,0.0,0.0
XRP_sma_20_100,0.5,0.3,0.2,0.0,0.0,0.0
XRP_sma_50_200,0.5,0.3,0.2,0.0,0.0,0.0
